In [4]:
import sqlite3
import json
from datetime import datetime
import pandas as pd

In [2]:
def init_db(db_name='property_vault.db'):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Create the main table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS listings (
            id TEXT PRIMARY KEY,           -- Unique ID from OLX/Storia
            platform TEXT,                 -- 'olx' or 'storia'
            title TEXT,
            url TEXT,
            price_eur REAL,
            description TEXT,
            raw_data TEXT,                 -- The full original JSON/Object
            scraped_at TIMESTAMP,
            is_processed BOOLEAN DEFAULT 0 -- Flag for LLM processing
        )
    ''')
    conn.commit()
    return conn

In [24]:
def save_olx_listing(db_name, listing_dict):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Extracting core fields from your dictionary
    listing_id = listing_dict.get('offer_id')
    title = listing_dict.get('title')
    url = listing_dict.get('url')
    price = listing_dict.get('ad_price')
    description = listing_dict.get('description')
    
    # Convert the whole dictionary to a string to store as 'raw_data'
    raw_json = json.dumps(listing_dict, ensure_ascii=False)
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        cursor.execute('''
            INSERT OR IGNORE INTO listings 
            (id, platform, title, url, price_eur, description, raw_data, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (listing_id, 'OLX', title, url, price, description, raw_json, now))
        conn.commit()
    except Exception as e:
        print(f"Error saving {listing_id}: {e}")
    finally:
        conn.close()

In [ ]:
def save_storia_listing(db_name, listing_dict):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Extracting core fields from your dictionary
    listing_id = listing_dict.get('id')
    title = listing_dict.get('title')
    url = listing_dict.get('url')
    price = listing_dict.get('price')
    description = listing_dict.get('description')
    
    # Convert the whole dictionary to a string to store as 'raw_data'
    raw_json = json.dumps(listing_dict['raw_json'], ensure_ascii=False)
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        cursor.execute('''
            INSERT OR IGNORE INTO listings 
            (id, platform, title, url, price_eur, description, raw_data, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (listing_id, 'Storia', title, url, price, description, raw_json, now))
        conn.commit()
    except Exception as e:
        print(f"Error saving {listing_id}: {e}")
    finally:
        conn.close()

In [ ]:
def migrate_olx_csv_to_db(df,db_name='property_vault.db'):
    # Ensure your CSV column names match the .get() calls in save_listing
    for _, row in df.iterrows():
        save_olx_listing(db_name, row.to_dict())
    print(f"✅ Migrated {len(df)} rows from given dataframe")

In [73]:
def migrate_storia_csv_to_db(df,db_name='property_vault.db'):
    # Ensure your CSV column names match the .get() calls in save_listing
    for _, row in df.iterrows():
        save_storia_listing(db_name, row.to_dict())
    print(f"✅ Migrated {len(df)} rows from given dataframe")

In [74]:
init_db('properties.db')

In [ ]:
olx_df = pd.read_csv('Date OLX.csv')
olx_df.drop(columns=list(olx_df.columns[:2]),inplace=True)
migrate_olx_csv_to_db(olx_df,'properties.db')
storia_df = pd.read_csv('Date Storia Procesate.csv')
storia_df.drop(columns=[storia_df.columns[0]],inplace=True)
migrate_storia_csv_to_db('properties.db')

✅ Migrated 42 rows from given dataframe
